In [3]:
# Load key and create the client 

import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv("../.env")
print("API key found:", os.getenv("ANTHROPIC_API_KEY") is not None)

client = Anthropic()
MODEL = "claude-haiku-5-5"

API key found: True


In [4]:
# Send a first message 

response = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role": "user", "content": "Name 3 French supermarket chains. Answer in one line."}
    ],
)

print(response.content[0].text)
print(f"Input tokens: {response.usage.input_tokens} | Output tokens: {response.usage.output_tokens}")

Carrefour, Leclerc, and Intermarché.
Input tokens: 28 | Output tokens: 21


In [5]:
# Import the generator 

%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

from src.generator import generate_dataset

df = generate_dataset()
print(df.shape)
df["category"].value_counts()

(4721, 4)


category
Restaurants            500
Shopping               500
Groceries              499
Cash Withdrawal        498
Leisure & Travel       452
Health                 443
Subscriptions          436
Transport              434
Bank Fees              270
Housing & Utilities    255
Transfers              237
Income                 197
Name: count, dtype: int64

In [6]:
# Category definitions and prompt 

import json
from src.generator import BASE_CATALOG

CATEGORY_DESCRIPTIONS = {
    "Groceries": "supermarkets, hypermarkets, convenience stores, organic shops, frozen food stores, independent local bakeries and butchers. NOT bakery-cafe chains like PAUL or BRIOCHE DOREE, which are Restaurants",
    "Restaurants": "restaurants, fast food chains, cafes, bakery-cafe chains, bars and food delivery apps",
    "Transport": "trains, metro, buses, ride-hailing, bike and scooter sharing, fuel stations, parking and motorway tolls",
    "Subscriptions": "digital subscriptions: video and music streaming, software, cloud storage, online news, AI tools. NOT phone or internet providers, NOT gyms",
    "Shopping": "retail stores and online shops: clothing, electronics, home and DIY, beauty, marketplaces, second-hand",
    "Housing & Utilities": "rent, property agencies, electricity, gas, water, internet and mobile phone operators, home and car insurance, property tax",
    "Health": "pharmacies, doctors, dentists, medical labs, opticians, physiotherapists, health insurance (mutuelles)",
    "Leisure & Travel": "cinemas, concerts, events, hotels, holiday rentals, airlines, video games, theme parks, museums, gyms and sports clubs",
    "Income": "French public bodies, pension funds and insurers that send money to individuals (benefits, pensions, refunds). NOT retail companies",
    "Cash Withdrawal": "French retail banks and their regional branches, as shown on ATM withdrawals",
    "Bank Fees": "names of bank fees and charges, as they appear on French bank statements",
}

PAYMENT_DESCRIPTIONS = {
    "card": "the customer pays by card",
    "debit": "the organization charges the customer by monthly SEPA direct debit",
    "transfer_out": "the customer sends money to the organization by bank transfer",
    "transfer_in": "the organization sends money to the customer by bank transfer",
    "withdrawal": "the customer withdraws cash from this bank's ATM",
    "fee": "the bank charges this fee to the customer",
}


def build_prompt(category, payment_type, existing, n=30):
    return f"""You are helping build a dataset of French bank transaction labels.

Category: {category}
Definition: {CATEGORY_DESCRIPTIONS[category]}
Payment type: {PAYMENT_DESCRIPTIONS[payment_type]}

List up to {n} real merchants or organizations active in France that fit this category and payment type.

Rules:
- Only include names you are confident really exist. Returning fewer names is better than inventing one.
- Each name must be a different brand or organization from the existing names below. Do not return store formats or variants of an existing brand (for example "CARREFOUR EXPRESS" when "CARREFOUR" exists).
- Write each name the way it typically appears on a French bank statement: uppercase, no accents, at most 25 characters.
- Only include names that clearly belong to this category. If a brand could also fit another category, leave it out.
- Existing names: {", ".join(existing)}

Return only a JSON array of strings, with no explanation and no markdown."""


print(build_prompt("Groceries", "card", BASE_CATALOG["Groceries"]["card"]))

You are helping build a dataset of French bank transaction labels.

Category: Groceries
Definition: supermarkets, hypermarkets, convenience stores, organic shops, frozen food stores, independent local bakeries and butchers. NOT bakery-cafe chains like PAUL or BRIOCHE DOREE, which are Restaurants
Payment type: the customer pays by card

List up to 30 real merchants or organizations active in France that fit this category and payment type.

Rules:
- Only include names you are confident really exist. Returning fewer names is better than inventing one.
- Each name must be a different brand or organization from the existing names below. Do not return store formats or variants of an existing brand (for example "CARREFOUR EXPRESS" when "CARREFOUR" exists).
- Write each name the way it typically appears on a French bank statement: uppercase, no accents, at most 25 characters.
- Only include names that clearly belong to this category. If a brand could also fit another category, leave it out.
- 

In [7]:
# Calling claude and reading the answer 

def ask_merchants(category, payment_type, existing, n=30):
    response = client.messages.create(
        model=MODEL,
        max_tokens=4000,
        thinking={"type": "disabled"},
        messages=[{"role": "user", "content": build_prompt(category, payment_type, existing, n)}],
    )

    if response.stop_reason == "max_tokens":
        raise ValueError(f"Response cut off for {category} / {payment_type}: increase max_tokens")

    text = "".join(block.text for block in response.content if block.type == "text").strip()
    text = text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text), response.usage


merchants, usage = ask_merchants("Groceries", "card", BASE_CATALOG["Groceries"]["card"])

print(f"{len(merchants)} merchants received")
print(merchants)
print(f"Input tokens: {usage.input_tokens} | Output tokens: {usage.output_tokens}")

16 merchants received
['HYPER U', 'MATCH', 'CORA', 'SYSTEME U', 'LECLERC EXPRESS', 'NETTO', 'ECO+', 'MY MARCHE', 'SPAR', 'VIVAL', 'PROXI', '8 A HUIT', 'LE MARCHE U', "BIO C' BON", 'ULTRA MARINE', 'OUI SNACK']
Input tokens: 513 | Output tokens: 116


In [8]:
# Print one merchant per line 

for merchant in merchants:
    print(merchant)

HYPER U
MATCH
CORA
SYSTEME U
LECLERC EXPRESS
NETTO
ECO+
MY MARCHE
SPAR
VIVAL
PROXI
8 A HUIT
LE MARCHE U
BIO C' BON
ULTRA MARINE
OUI SNACK


In [9]:
# Check the new filter

from src.generator import build_catalog, merchant_key

print(merchant_key("G 20"), merchant_key("G20"), merchant_key("E.Leclerc"))

test_catalog = build_catalog({"Groceries": {"card": ["G 20", "PAUL", "Netto"]}})
print(test_catalog["Groceries"]["card"][-3:])

G20 G20 ELECLERC
['G20', 'LA VIE CLAIRE', 'NETTO']


In [15]:
from pathlib import Path

output_path = Path("../data/llm_merchants.json")

if output_path.exists():
    print(f"{output_path} already exists: skipping the API calls.")
    print("Delete the file if you really want to generate new suggestions.")
else:
    llm_merchants = {}
    total_input, total_output = 0, 0

    for category in CATEGORY_DESCRIPTIONS:
        llm_merchants[category] = {}
        for payment_type, existing in BASE_CATALOG[category].items():
            try:
                merchants, usage = ask_merchants(category, payment_type, existing)
            except Exception as error:
                print(f"{category:<22}{payment_type:<14}FAILED: {error}")
                continue

            llm_merchants[category][payment_type] = merchants
            total_input += usage.input_tokens
            total_output += usage.output_tokens
            print(f"{category:<22}{payment_type:<14}{len(merchants)} merchants")

    print(f"\nTotal tokens: {total_input} input | {total_output} output")
    output_path.write_text(json.dumps(llm_merchants, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Saved to {output_path}")

..\data\llm_merchants.json already exists: skipping the API calls.
Delete the file if you really want to generate new suggestions.


In [16]:
# How many suggestion does the filter accept ? 

from src.generator import build_catalog, strip_accents

llm_merchants = json.loads(Path("../data/llm_merchants.json").read_text(encoding="utf-8"))

base = build_catalog()
full = build_catalog(llm_merchants)
new_merchants = {}

for category, groups in llm_merchants.items():
    for payment_type, suggested in groups.items():
        n_before = len(base[category].get(payment_type, []))
        added = full[category][payment_type][n_before:]
        new_merchants[(category, payment_type)] = added

        added_set = set(added)
        rejected = [m for m in suggested if strip_accents(m.upper()).strip() not in added_set]
        print(f"{category:<22}{payment_type:<14}{len(suggested):>3} suggested | {len(added):>3} accepted")
        if rejected:
            print(f"{'':<36}rejected: {', '.join(rejected)}")

Groceries             card           30 suggested |  30 accepted
Restaurants           card           30 suggested |  28 accepted
                                    rejected: BURGER KING, PAUL
Transport             card           27 suggested |  27 accepted
Transport             debit          30 suggested |  18 accepted
                                    rejected: LIME, TIER, ZITY, TOTALENERGIES, SANEF, APRR, SAEMES, RATP, OUIGO, BLABLACAR, LEASYS, SAGS
Subscriptions         card           30 suggested |  27 accepted
                                    rejected: DEEZER, CANAL+, LE MONDE
Subscriptions         debit          29 suggested |   9 accepted
                                    rejected: SPOTIFY, DISNEY PLUS, AMAZON PRIME, APPLE.COM/BILL, MICROSOFT 365, ADOBE, DROPBOX, ICLOUD, YOUTUBE PREMIUM, NOTION, CANVA, 1PASSWORD, NORDVPN, MEDIAPART, LE FIGARO, NYTIMES, TWITCH, CRUNCHYROLL, DROPBOX, PARAMOUNT PLUS
Shopping              card           35 suggested |  34 accepted
        

In [17]:
# Human review of the suspicious categories 

import textwrap

for (category, payment_type), names in new_merchants.items():
    print(f"=== {category} / {payment_type} ({len(names)}) ===")
    print(textwrap.fill(", ".join(names), width=110))
    print()

=== Groceries / card (30) ===
HYPER U, MAXI MARCHE, FAMILY MARCHE, COCCINELLE SUPERMARCHE, SIMPLY MARKET, SPAR, MATCH, CORA, HYPER CASINO,
DIA, NETTO, ECO+, LIDL PLUS, SUPER CASINO, OUI SERVICE, PROXI, VIVAL, 8 A HUIT, CHAMPION, HYPER AUCHAN,
MIGROS, ATAC, RELAY, FRANPRIX PROXIMITE, CASINO PROXIMITE, BOULANGERIE, BOUCHERIE, FROMAGERIE, EPICERIE FINE,
NATURE & DECOUVERTES

=== Restaurants / card (28) ===
SUBWAY, NORDSEE, LA MIE CALINE, VITY, FLUNCH, LEON DE BRUXELLES, PIZZA HUT, TACO BELL, BLEU BLANC ROUGE, CAFE
COSTA, EXKI, BOJANGLES, KIBO, POKAWA, BEAVER, GOLDEN BURGER, BISTRO REGENT, LA GRANDE EPICERIE, CHIPOTLE,
NORMA, BERTHILLON, CHAMPS CAFE, CAFE PUSHKIN, GREGOIRE TACOS, TABLE DE FRANCE, STREET FOOD, TOUS LES JOURS,
BOUILLON CHARTIER

=== Transport / card (27) ===
FREE TO X, TIER, VOI, ZITY, AUTOLIB, CAR2GO, LEASYS, ENTERPRISE, AVIS, HERTZ, EUROPCAR, ADA, TAXI G7,
MARGUERITE, ARVAL, STAR, TOTAL ACCESS, AXA PARKING, SAGS, ALLIANCE CARBURANTS, LECLERC CARBURANT, CARTE
CARBURANT, APR

In [20]:
# Apply the exclusions and save the reviewed file 

from src.generator import merchant_key

EXCLUDED = {
    ("Groceries", "card"): [
        "NATURE & DECOUVERTES", "RELAY",              # Shopping, not groceries
        "ECO+",                                       # product line, not a merchant
        "CHAMPION", "ATAC", "DIA", "SIMPLY MARKET",  # banners that no longer exist
    ],
    ("Restaurants", "card"): [
        "LA GRANDE EPICERIE", "NORMA",  # grocery stores
    ],
    ("Transport", "card"): [
        "AXA PARKING",          # AXA is an insurer: misleading name
        "ECOLOGIC",             # electronics recycling, not transport
        "FREE TO X",            # unidentified, and FREE is a telecom brand (Housing)
        "AUTOLIB", "CAR2GO",    # services that shut down
    ],
    ("Transport", "debit"): [
        "BOLT", "BOLT CARPOOL", "BOLT DRIVE",  # paid by card, not debit
        "CITYSCOOT",                           # shut down in 2023
        "AIRPARIF",                            # air quality agency, not transport
        "STATION F VELO",                      # unidentified service
        "CARTE BLEUE PARKING",                 # "Carte Bleue" is a card brand: misleading
    ],
    ("Subscriptions", "card"): [
        "PLAYSTATION PLUS", "NINTENDO SWITCH ONLINE", "XBOX GAME PASS",  # gaming brands are Leisure
    ],
    ("Shopping", "card"): [
        "NORAUTO",  # car maintenance: ambiguous between Shopping and Transport
    ],
    ("Housing & Utilities", "debit"): [
        "AVIVA ASSURANCES",  # renamed Abeille Assurances in 2021
    ],
    ("Housing & Utilities", "transfer_out"): [
        "LA FRANCAISE DES JEUX",  # national lottery: Leisure, not housing
    ],
    ("Health", "card"): [
        "L'OCCITANE",  # cosmetics shop: Shopping
    ],
    ("Health", "debit"): [
        "COVEA",  # insurance group (MAAF, MMA, GMF): mostly Housing, ambiguous
    ],
    ("Leisure & Travel", "card"): [
        "POMME DE PIN",  # probably "Pomme de Pain", a sandwich chain: Restaurants
    ],
    ("Leisure & Travel", "debit"): [
        "ORANGE CINEMA SERIES",  # streaming service: Subscriptions
        "FORUM DES HALLES",      # shopping mall: Shopping
        "LE MONDE DU SPORT",     # sounds like a sports shop, and LE MONDE is Subscriptions
        "MARINELAND",            # closed in 2025
    ],
    ("Income", "transfer_in"): [
        "URSSAF",       # collects contributions, rarely pays individuals
        "PREFECTURE",   # doesn't pay individuals
        "POLE EMPLOI",  # renamed France Travail in 2024
        "RSI",          # abolished
    ],
    ("Cash Withdrawal", "withdrawal"): [
        "ING DIRECT",         # left French retail banking
        "CREDIT DU NORD",     # merged into Societe Generale in 2023
        "CREDIT LYONNAIS",    # called LCL since 2005
        "CREDIT FONCIER",     # mortgage lender, no retail ATMs
        "CAISSE DES DEPOTS",  # public institution, no ATMs
        "ORANGE BANK",        # customers transferred to another bank
    ],
}

reviewed = {}
n_removed = 0

for category, groups in llm_merchants.items():
    reviewed[category] = {}
    for payment_type, merchants in groups.items():
        excluded_keys = {merchant_key(m) for m in EXCLUDED.get((category, payment_type), [])}
        kept = [m for m in merchants if merchant_key(m) not in excluded_keys]
        n_removed += len(merchants) - len(kept)
        reviewed[category][payment_type] = kept

reviewed_path = Path("../data/llm_merchants_reviewed.json")
reviewed_path.write_text(json.dumps(reviewed, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Removed {n_removed} merchants. Saved to {reviewed_path}")

Removed 44 merchants. Saved to ..\data\llm_merchants_reviewed.json


In [21]:
# The payoff, before vs after 

import pandas as pd
from src.generator import generate_dataset

reviewed = json.loads(reviewed_path.read_text(encoding="utf-8"))

df_before = generate_dataset()
df_after = generate_dataset(extra_merchants=reviewed)

comparison = pd.DataFrame({
    "rows_before": df_before["category"].value_counts(),
    "rows_after": df_after["category"].value_counts(),
    "merchants_before": df_before.groupby("category")["merchant"].nunique(),
    "merchants_after": df_after.groupby("category")["merchant"].nunique(),
})

conflicts = (df_after.groupby("label")["category"].nunique() > 1).sum()
print(f"Total rows: {len(df_before)} -> {len(df_after)}")
print(f"Labels assigned to more than one category: {conflicts}")

df_after.to_csv("../data/raw/transactions.csv", index=False)

comparison.sort_values("merchants_after")

Total rows: 4721 -> 4960
Labels assigned to more than one category: 0


,rows_before,rows_after,merchants_before,merchants_after
category,,,,
Bank Fees,270,313,10,30
Cash Withdrawal,498,499,10,34
Groceries,499,499,19,42
Transfers,237,235,45,45
Restaurants,500,499,21,47
Income,197,267,21,47
Subscriptions,436,448,17,50
Leisure & Travel,452,455,21,55
Transport,434,449,22,55


In [18]:
groups = list(new_merchants.items())

for (category, payment_type), names in groups[5:10]:
    print(f"=== {category} / {payment_type} ({len(names)}) ===")
    print(textwrap.fill(", ".join(names), width=110))
    print()

=== Subscriptions / debit (9) ===
NETFLIX, GOOGLE STORAGE, DEEPL, CHATGPT, OPENAI, LIBERATION, L EXPRESS, APPLE MUSIC, TIDAL

=== Shopping / card (34) ===
LA REDOUTE, MANGO, JULES, KARL MARC JOHN, BRICO DEPOT, BRICOMARCHE, MAISONS DU MONDE, BUT, CONFORAMA, SAMSUNG
STORE, MEDIA MARKT, LEGO STORE, GALERIES LAFAYETTE, PRINTEMPS, BOOSTER, YVES ROCHER, DOUGLAS, RUE DU COMMERCE,
RAKUTEN, EBAY, ALIEXPRESS, SHEIN, ASOS, BERSHKA, PULL AND BEAR, JD SPORTS, INTERSPORT, DIMSPORT, GO SPORT,
OKAIDI, JACADI, MISTERGOOD, ELECTRO DEPOT, NORAUTO

=== Housing & Utilities / debit (29) ===
NRJ ENERGIE, PLENITUDE, ENI GAZ ET ELECTRICITE, VATTENFALL FRANCE, ENERCOOP, ENERGIE DIRECTE, MOBIL ENERGIE,
SYDEV EAU, SAUR, AQUA DE FRANCE, LYDIC, NORMANDIE ASSURANCES, GROUPAMA, MATMUT, GENERALI FRANCE, ALLIANZ
FRANCE, AVIVA ASSURANCES, HISCOX FRANCE, LEGAL ASSURANCE, ASSURANCE ASPIRE, BOUYGUES BOX, NRJ MOBILE, LEBARA
FRANCE, LYCAMOBILE FRANCE, B&YOU, SPUTNIK MOBILE, NRJ MOBILE FRANCE, CITYA IMMOBILIER, ORPI

=== Hous

In [19]:
for (category, payment_type), names in groups[10:15]:
    print(f"=== {category} / {payment_type} ({len(names)}) ===")
    print(textwrap.fill(", ".join(names), width=110))
    print()

=== Health / debit (21) ===
AXA SANTE, MACIF MUTUELLE, GROUPAMA SANTE, AG2R LA MONDIALE, APRIL SANTE, SWISS LIFE SANTE, COVEA, SANTECLAIR,
GENERALE OPTIQUE, CENTRE OPTIQUE, DOCTEUR BONNET, CERBA, EUROFINS BIOLOGIE, BIOMNIS, LABORATOIRE ALPHA,
DENTEGRA, CAPIO, VITALIA, KINE PLUS, HARMONIE SANTE, SANTE PHARMA

=== Leisure & Travel / card (25) ===
CGR CINEMAS, LE GRAND REX, ZENITH PARIS, OLYMPIA, ACCOR HOTELS, IBIS HOTEL, NOVOTEL, HOLIDAY INN, HOTELS
KYRIAD, VUELING, VOLOTEA, HOP AIR FRANCE, XBOX, EPIC GAMES, GOOGLE PLAY GAMES, KEEPCOOL, MUSEE D ORSAY, CENTRE
POMPIDOU, CITE DES SCIENCES, PUY DU FOU, FUTUROSCOPE, WALIBI, ASTERIX PARC, VACANCES BLEUES, POMME DE PIN

=== Leisure & Travel / debit (12) ===
UGC CINE CITE, GAUMONT, LE CLUB DE FITNESS, ORANGE CINEMA SERIES, EUROPA PARC, MARINELAND, IBIS HOTELS,
MERCURE HOTELS, FORUM DES HALLES, ZENITH, LE MONDE DU SPORT, NEO FITNESS

=== Income / transfer_in (30) ===
URSSAF, CARSAT, CNAV, MSA, AGIRC ARRCO, CNRACL, POLE EMPLOI, CPAM ILE DE FRANCE,